# MAI391 — Red Wine Quality: Dataset & Preprocessing (Khoa)

**Compatible:** Jupyter Notebook / VS Code / Colab (không yêu cầu `google.colab`).  
**Run:** mở `notebooks/Khoa_Dataset_Preprocessing.ipynb` và chọn **Run All**.  
**Input:** `data/winequality-red.csv` (nếu chưa có, notebook thử tải từ UCI qua Internet).  
**Output:** `data/`, `reports/`, `models/` ở **repository root**, không phụ thuộc working directory.

**Group's fixed decisions:** 1,599 rows, 11 ordered numerical features, target `quality`, keep duplicates, `test_size=0.2`, `random_state=42`; baseline không scale. Có tùy chọn StandardScaler fit trên train. Không huấn luyện mô hình tại đây.


## 1. Import thư viện và tự tìm repository root

In [ ]:
from pathlib import Path
import hashlib
import json
import urllib.request

import numpy as np
import pandas as pd
import joblib
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

RANDOM_STATE = 42
TEST_SIZE = 0.2
SOURCE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

# Notebook ở repo/notebooks hoặc chạy từ repo root đều tìm được repo.
# __file__ không có trong Jupyter nên sử dụng Path.cwd().
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for parent in (start, *start.parents):
        if (parent / "data").is_dir() and (parent / "notebooks").is_dir():
            return parent
    # Cho trường hợp mới clone chưa có đủ thư mục; chỉ dùng nếu ở đúng notebooks/.
    if start.name == "notebooks":
        return start.parent
    if (start / "notebooks").is_dir():
        return start
    raise FileNotFoundError(
        "Không tìm thấy project root. Mở notebook bên trong repo có "
        "thư mục 'data/' và 'notebooks/', hoặc chạy Jupyter/VS Code từ repo root."
    )

ROOT = find_project_root(Path.cwd())
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports"
MODELS_DIR = ROOT / "models"
for directory in (DATA_DIR, REPORTS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

files_written = []

print("Project root:", ROOT)
print("CSV location:", DATA_DIR / "winequality-red.csv")
print("Pandas version:", pd.__version__)

## 2. Đọc CSV UCI từ `data/` (không cần Google Colab)
Nếu file chưa có, notebook thử tải về `data/winequality-red.csv`. Máy không có mạng: tải file UCI thủ công rồi đặt đúng đường dẫn này.

In [ ]:
csv_path = DATA_DIR / "winequality-red.csv"
if not csv_path.exists():
    try:
        with urllib.request.urlopen(SOURCE_URL, timeout=30) as response:
            csv_path.write_bytes(response.read())
        files_written.append(csv_path)
        print("Downloaded UCI dataset to:", csv_path)
    except Exception as exc:
        raise FileNotFoundError(
            f"Không thấy {csv_path}. Tải CSV tại {SOURCE_URL}, "
            "lưu tên winequality-red.csv vào thư mục data/, rồi Run All lại."
        ) from exc
else:
    print("Reading local CSV:", csv_path)

source_sha256 = hashlib.sha256(csv_path.read_bytes()).hexdigest()
df = pd.read_csv(csv_path, sep=";")
print("Shape:", df.shape)
print("Source SHA256:", source_sha256)
display(df.head())

## 3. Kiểm tra cấu trúc, missing values và duplicate rows
Giữ 240 bản ghi trùng theo thống nhất của nhóm. Các dòng trùng có thể xuất hiện ở cả train/test, cần nêu hạn chế này khi trình bày.

In [ ]:
EXPECTED_COLUMNS = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol", "quality"
]
assert df.columns.tolist() == EXPECTED_COLUMNS, "Sai tên/thứ tự cột (kiểm tra sep=';')."
assert df.shape == (1599, 12), "CSV không phải bản UCI Red Wine Quality gốc 1599x12."
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns), "Có cột không phải số."

missing_by_column = df.isna().sum()
duplicate_count = int(df.duplicated().sum())
unique_row_count = int(len(df.drop_duplicates()))
column_quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": missing_by_column,
    "missing_pct": (missing_by_column / len(df) * 100).round(2)
})
assert int(missing_by_column.sum()) == 0, "Có missing values: cần kiểm tra dữ liệu."
assert duplicate_count == 240, "Duplicate count khác dữ liệu gốc: cần kiểm tra file."
print("Missing values:", int(missing_by_column.sum()))
print("Duplicate rows (retained):", duplicate_count)
print("Unique rows:", unique_row_count)
display(column_quality)
column_quality.to_csv(REPORTS_DIR / "column_quality_report.csv", encoding="utf-8-sig")
files_written.append(REPORTS_DIR / "column_quality_report.csv")


## 4. Thống kê mô tả và phân phối `quality`

In [ ]:
descriptive_statistics = df.describe().T.rename_axis("variable").reset_index()
descriptive_statistics.to_csv(REPORTS_DIR / "descriptive_statistics.csv", index=False, encoding="utf-8-sig")
files_written.append(REPORTS_DIR / "descriptive_statistics.csv")
quality_distribution = (
    df["quality"].value_counts().sort_index().rename_axis("quality")
      .reset_index(name="count")
)
quality_distribution["percentage"] = (
    quality_distribution["count"] / len(df) * 100
).round(2)
quality_distribution.to_csv(REPORTS_DIR / "quality_distribution.csv", index=False, encoding="utf-8-sig")
files_written.append(REPORTS_DIR / "quality_distribution.csv")
display(descriptive_statistics.round(4))
display(quality_distribution)

## 5. X (11 features) và y (`quality`); giữ đúng thứ tự cột

In [ ]:
FEATURE_COLUMNS = EXPECTED_COLUMNS[:-1]
TARGET_COLUMN = "quality"
X = df.loc[:, FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()
assert X.shape == (1599, 11) and y.shape == (1599,)
assert TARGET_COLUMN not in X.columns
(REPORTS_DIR / "feature_order.json").write_text(
    json.dumps({"features": FEATURE_COLUMNS, "target": TARGET_COLUMN}, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
files_written.append(REPORTS_DIR / "feature_order.json")
print("X:", X.shape, "| y:", y.shape)
print("Feature order:", FEATURE_COLUMNS)

## 6. Chia train/test 80/20; random_state=42
Không xóa duplicates trước khi chia. Xuất `train.csv`, `test.csv` giữ cả target để M4/M5 sử dụng lại, không cần chia lần nữa.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
assert (X_train.shape, X_test.shape) == ((1279, 11), (320, 11))
assert len(X_train.index.intersection(X_test.index)) == 0
train_df = X_train.assign(quality=y_train)
test_df = X_test.assign(quality=y_test)
train_df.to_csv(DATA_DIR / "train.csv", index=False)
files_written.append(DATA_DIR / "train.csv")
test_df.to_csv(DATA_DIR / "test.csv", index=False)
files_written.append(DATA_DIR / "test.csv")
pd.Series(X_train.index, name="original_row_index").to_csv(REPORTS_DIR / "train_indices.csv", index=False)
files_written.append(REPORTS_DIR / "train_indices.csv")
pd.Series(X_test.index, name="original_row_index").to_csv(REPORTS_DIR / "test_indices.csv", index=False)
files_written.append(REPORTS_DIR / "test_indices.csv")

train_rows = pd.MultiIndex.from_frame(train_df)
test_rows = pd.MultiIndex.from_frame(test_df)
test_matches_train = test_rows.isin(train_rows)
overlap_test_rows = int(test_matches_train.sum())
overlap_unique = int(test_rows[test_matches_train].nunique())
print(f"Test rows matching a full train record: {overlap_test_rows}/{len(test_df)}")
print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("Unique duplicate full-record values across train/test:", overlap_unique)

## 7. StandardScaler tùy chọn — fit TRAIN, transform TEST
Baseline dùng dữ liệu gốc. File scaler trong `models/` chỉ dùng nếu nhóm thống nhất chạy thí nghiệm chuẩn hóa.

In [ ]:
# Option 1 (baseline): X_train và X_test KHÔNG scale.
# Option 2: StandardScaler fit chỉ trên X_train.
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=FEATURE_COLUMNS, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=FEATURE_COLUMNS, index=X_test.index
)
assert np.allclose(X_train_scaled.mean().to_numpy(), 0, atol=1e-10)
joblib.dump(scaler, MODELS_DIR / "standard_scaler.pkl")
files_written.append(MODELS_DIR / "standard_scaler.pkl")

# Ví dụ pipeline được phép dùng nếu nhóm muốn bật scaling ở M4:
optional_scaled_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])
# optional_scaled_pipeline.fit(X_train, y_train)
# y_pred = optional_scaled_pipeline.predict(X_test)
print("Scaled train mean max absolute:", X_train_scaled.mean().abs().max())
print("Saved optional scaler:", MODELS_DIR / "standard_scaler.pkl")

## 8. Xuất inspection report và manifest theo đường dẫn GitHub
Không có `files.download()` hoặc phụ thuộc Google Colab. Các file được ghi trực tiếp vào thư mục repo.

In [ ]:
manifest = {
    "dataset": "UCI Wine Quality — Red",
    "source_url": SOURCE_URL,
    "source_sha256": source_sha256,
    "shape": list(df.shape),
    "features": FEATURE_COLUMNS,
    "target": TARGET_COLUMN,
    "missing_cells": int(missing_by_column.sum()),
    "duplicate_rows": duplicate_count,
    "unique_rows": unique_row_count,
    "split": {
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "train_size": len(X_train),
        "test_size_rows": len(X_test)
    },
    "duplicate_policy": "retained for baseline",
    "test_rows_matching_train_full_records": overlap_test_rows,
    "unique_full_records_shared_between_train_test": overlap_unique,
    "scaler": "models/standard_scaler.pkl (optional; fit on train only)"
}
(REPORTS_DIR / "preprocessing_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
files_written.append(REPORTS_DIR / "preprocessing_manifest.json")
report_text = f"""# MAI391 — Dataset inspection (Khoa)

- Source: {SOURCE_URL}
- Source SHA-256: `{source_sha256}`
- Dataset shape: **{df.shape[0]} × {df.shape[1]}**
- Input features: **{len(FEATURE_COLUMNS)}**; target: **`quality`**
- Missing cells: **{int(missing_by_column.sum())}**
- Duplicate rows: **{duplicate_count}**, retained for baseline
- Training samples: **{len(X_train)}**
- Testing samples: **{len(X_test)}**
- Split: **80/20**, `random_state=42`
- Quality observed range: **{df['quality'].min()}–{df['quality'].max()}**
- Test rows matching a full training record: **{overlap_test_rows}/{len(test_df)}**
- Unique full-record values present in both split sets: **{overlap_unique}** (limitation)
- StandardScaler optional, fitted on train only

Files: `data/train.csv`, `data/test.csv`, `reports/feature_order.json`,
`reports/descriptive_statistics.csv`, `reports/quality_distribution.csv`,
`reports/preprocessing_manifest.json`, `models/standard_scaler.pkl`.
"""
(REPORTS_DIR / "inspection_report.md").write_text(report_text, encoding="utf-8")
files_written.append(REPORTS_DIR / "inspection_report.md")
print(report_text)
print("Files written:")
for output_file in sorted(files_written):
    print(" -", output_file.relative_to(ROOT).as_posix())